# Offline Repair Model Training for Hybrid ALNS

---

## I. Objective and Scope

This notebook specifies and executes the **offline supervised-learning pipeline** used to train the repair model consumed by the hybrid ALNS solver.

Given ALNS states and candidate insertions, the learning objective is to estimate a score function
$f_\theta(\phi(x))$ that ranks feasible repair actions by expected improvement quality. The trained artifact is serialized as `../models/repair_model.pkl`.

This notebook covers the offline stage:
- generation of labeled state-action examples,
- training/validation of the ranking surrogate,
- model persistence for downstream search-time inference.


## II. Data-Generation Protocol

Let $x_t$ denote a partially destroyed solution state at ALNS iteration $t$, and let $\mathcal{A}(x_t)$ be the set of feasible repair actions. Data generation samples tuples
$(x_t, a, y)$, where $a \in \mathcal{A}(x_t)$ and label $y$ encodes relative quality/acceptability under the training objective used by `collect_alns_states.py`.

Important protocol choices for statistical robustness:
- instance-size diversity (`--n-min`, `--n-max`),
- sufficient trajectory coverage (`--instances`),
- controlled negative sampling (`--max-negatives`),
- reproducibility (`--seed`).


## III. Training Configuration

The following configuration is the default high-quality recipe for tabular ALNS features:
- model family: gradient-boosted trees (`--model xgb`),
- boosting depth/learning tradeoff: `--max-depth 6`, `--learning-rate 0.05`,
- ensemble capacity: `--n-estimators 400`,
- stochastic regularization: `--subsample 0.9`, `--colsample-bytree 0.8`.


In [1]:
!python collect_alns_states.py \
    --instances-dir ../instances \
    --output states_dataset.csv \
    --n-instances 500 \
    --iters 300 \
    --seed 42

usage: collect_alns_states.py [-h] --model-path MODEL_PATH
                              [--instances INSTANCES] [--n-min N_MIN]
                              [--n-max N_MAX] [--max-negatives MAX_NEGATIVES]
                              [--iterations ITERATIONS] [--seed SEED]
                              [--output OUTPUT]
collect_alns_states.py: error: the following arguments are required: --model-path


In [2]:
!python train_repair_model.py \
    --data states_dataset.csv \
    --out ../models/repair_model.pkl \
    --model xgb \
    --n-estimators 400 \
    --max-depth 6 \
    --learning-rate 0.05 \
    --subsample 0.9 \
    --colsample-bytree 0.8 \
    --seed 42

usage: train_repair_model.py [-h] [--instances INSTANCES] [--n-min N_MIN]
                             [--n-max N_MAX] [--max-negatives MAX_NEGATIVES]
                             [--destroy-fraction DESTROY_FRACTION]
                             [--seed SEED] [--workers WORKERS]
                             [--output OUTPUT] [--augment-with PKL]
                             [--prewarm-numba]
train_repair_model.py: error: unrecognized arguments: --data states_dataset.csv --model xgb --n-estimators 400 --max-depth 6 --learning-rate 0.05 --subsample 0.9 --colsample-bytree 0.8


## IV. Output Contract and Hand-Off to Runtime

Successful execution must produce `../models/repair_model.pkl`. The runtime notebooks assume this artifact is present and loadable.

Before proceeding, verify:
1. training finished without exceptions,
2. validation/test metrics are reported,
3. output model path exists and matches runtime command arguments.
